# ORION — 사전등록 캠페인 실행 (Google Colab)

이 노트북은 `RUN_STEPS.md`의 **STEP 0–5**를 Colab에서 한 번에 실행합니다.

**시작 전 (중요):** 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU** 로 설정하세요.

- STEP 0–2, 4 는 GPU 없이도 동작(점검용), **STEP 3 은 GPU 필요**.
- 코드 작성은 필요 없습니다. 위에서부터 각 셀을 순서대로 실행(▶)하면 됩니다.
- 결과물 `records.jsonl` 은 STEP 5 에서 자동 다운로드됩니다.


## STEP 0 — 저장소 가져오기


In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('/content/orion'):
    !git clone https://github.com/leemgs/orion.git
os.chdir('/content/orion')
!git checkout main && git pull origin main
print('cwd =', os.getcwd())

## STEP 1 — 의존성 (Colab에는 torch가 이미 설치돼 있음)


In [ ]:
!pip -q install numpy pytest
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  GPU가 없습니다. 런타임 → 런타임 유형 변경 → GPU 로 바꾼 뒤 STEP 0부터 다시 실행하세요.')

## STEP 2 — 배관 점검 (GPU 불필요)

테스트가 통과하고, dry-run 레코드가 분석기에서 **거부**되면 정상입니다(안전장치).


In [ ]:
!python -m pytest code/tests -q
print('\n--- dry-run 생성 ---')
!python code/experiments/prereg_harvest.py --dry-run -o records_dryrun.jsonl
print('\n--- 분석기가 dry-run 을 거부해야 정상 ---')
!python code/experiments/analyze_prereg.py records_dryrun.jsonl || echo 'OK: dry-run 거부됨(정상)'

## STEP 3 — 실제 측정 (GPU 필요, 코드 작성 없음)

`torch-reference` 백엔드가 이 Colab GPU에서 실제 연산·전송을 CUDA 이벤트로 측정합니다.
`--arch` 는 실제 GPU 이름으로 자동 기록됩니다. (그리드 전체 측정이라 **수 분** 소요될 수 있습니다.)


In [ ]:
import torch, re
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다. STEP 1 안내를 보세요.'
arch = re.sub(r'[^A-Za-z0-9.-]+', '-', torch.cuda.get_device_name(0)).strip('-')
print('기록될 arch =', arch)
!python code/experiments/prereg_harvest.py \
  --backend torch-reference \
  --machine-id colab-1 --arch {arch} --model ref-workload \
  --runs 5 -o records.jsonl
print('\n생성된 줄 수:')
!wc -l records.jsonl

### (선택) 아키텍처 2종 이상 모으기

심사 요건은 가속기 아키텍처 **≥2종**입니다. Colab에서 런타임 유형의 GPU를 바꿔
(예: T4 → L4/A100) STEP 0–3 을 다시 실행하면 매번 다른 `records.jsonl` 이 나옵니다.
각 실행 후 아래처럼 이름을 바꿔 두었다가, 마지막에 합치세요.

```python
# 예: 첫 실행 후
!cp records.jsonl records_T4.jsonl
# 다른 GPU 런타임에서 다시 STEP 0–3 실행 후
!cp records.jsonl records_L4.jsonl
# 합치기
!cat records_*.jsonl > records.jsonl
```


## STEP 4 — 검증 + 미리보기 (GPU 불필요)


In [ ]:
import sys, json
sys.path.insert(0, 'code')
from experiments.prereg_harvest import validate_record
n = 0
for line in open('records.jsonl'):
    validate_record(json.loads(line)); n += 1
print(f'schema OK: {n} rows')
print('\n--- 사전등록 분석 (held-out) ---')
!python code/experiments/analyze_prereg.py records.jsonl --split heldout

## STEP 5 — 결과 내려받기 → 저에게 전달

아래 셀이 `records.jsonl` 을 로컬로 다운로드합니다. 그 파일을 저장소에 커밋하시거나,
채팅에 첨부/경로를 알려주시면 이후 분석·표·그림·본문 반영·병합은 제가 합니다.


In [ ]:
from google.colab import files
files.download('/content/orion/records.jsonl')

### (선택) Colab에서 바로 커밋하려면 (GitHub 토큰 필요)

```python
TOKEN = ''  # GitHub personal access token (repo 권한). 노출 주의!
!git config --global user.email 'leemgs@gmail.com'
!git config --global user.name 'Geunsik Lim'
!mkdir -p code/results/prereg_campaign && cp records.jsonl code/results/prereg_campaign/records.jsonl
!git add code/results/prereg_campaign/records.jsonl && git commit -m 'Add preregistered campaign records'
!git push https://{TOKEN}@github.com/leemgs/orion.git main
```

---
**더 강한 근거(선택):** `code/experiments/prereg_backends.py` 의 `VLLMBackend` 등
`measure()` 를 채우면 실제 서빙 스택으로 측정할 수 있습니다. `handoff_checklist.md` 참고.
